In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------- Affine --------------------
class AffineTransform():
    def __init__(self, do_scale=False, do_rotation=True):
        self.do_scale = do_scale
        self.do_rotation = do_rotation
        self.scale = 1

    def fit(self, source_points, target_points):
        assert len(source_points) == len(target_points)
        self.S_centroid = np.mean(source_points, axis=0)
        self.T_centroid = np.mean(target_points, axis=0)
        self.source_points_centered=source_points-self.S_centroid
        self.target_points_centered=target_points-self.T_centroid
        H = np.dot(np.transpose(self.source_points_centered),  self.target_points_centered)
        rank_H = np.linalg.matrix_rank(H)

        if not self.do_rotation:
            self.rotation_matrix= np.eye(source_points.shape[1])
        elif rank_H < source_points.shape[1]:
            self.rotation_matrix= np.eye(source_points.shape[1])
        else:   
            U, S, Vt = np.linalg.svd(H)
            V=Vt.T
            self.rotation_matrix = V @ U.T 
            if np.linalg.det(self.rotation_matrix)<0:
                V[:,-1]*= -1
                self.rotation_matrix= V @ U.T

        if self.do_scale:
            source_rotated=np.transpose(self.rotation_matrix @ np.transpose((self.source_points_centered)))
            self.scale = np.sum(source_rotated * self.target_points_centered) / np.sum(source_rotated**2)
        self.translation=self.T_centroid-self.S_centroid
        
    def predict(self, x):
        transported_x= self.scale* (x-self.S_centroid) @ np.transpose(self.rotation_matrix) + self.T_centroid
        return transported_x

# -------------------- GP --------------------
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel

class GaussianProcess():
    def __init__(self, kernel, alpha=1e-10, optimizer='fmin_l_bfgs_b', n_restarts_optimizer=5, n_targets=None):
        if optimizer != None:
            self.gp = GaussianProcessRegressor(kernel=kernel, alpha=alpha, optimizer=optimizer ,n_restarts_optimizer=n_restarts_optimizer)
        else:
            self.gp = GaussianProcessRegressor(kernel=kernel, alpha=alpha, optimizer=optimizer)      
        self.kernel=kernel
        self.alpha=alpha
        self.is_residual = True
    def fit(self, X, Y):
            self.X=X; self.Y=Y
            self.n_features=np.shape(self.X)[1]
            self.n_samples=np.shape(self.X)[0]
            self.n_outputs=np.shape(self.Y)[1]
            self.gp.fit(self.X,self.Y)
            self.kernel= self.gp.kernel_

    def predict(self,x, return_std=False, return_cov=False):
        if return_std==True:
            y, std = self.gp.predict(x, return_std=True)
            return np.array(y), np.array(std)
        if return_cov==True:
            y, cov = self.gp.predict(x, return_cov=True)
            return np.array(y), np.array(cov)
        else:
            y=self.gp.predict(x, return_std=False)
            return np.array(y)

# -------------------- Curve --------------------
y = np.linspace(0, 1, 200)
x = 2*y**3 + 1*y**2
curve = np.c_[x, y]

# Original keypoints
S = np.array([[x[0], y[0]], [2, 0.6], [x[-1], y[-1]]])

# Circle boundary around [2,0.6]
mid_center = np.array([2.0, 0.6])
r = 0.15
theta = np.linspace(0, 2*np.pi, 20)
circle_pts = np.c_[mid_center[0] + r*np.cos(theta), mid_center[1] + r*np.sin(theta)]

# Sweep last keypoint
last_targets = np.linspace(0.9, -5.0, 100)

kernel = ConstantKernel(1.0) * RBF(length_scale=0.6) + WhiteKernel(noise_level=1e-10)

plt.figure(figsize=(9,9))
plt.plot(curve[:,0], curve[:,1], color='gray', linestyle='--', label='source curve')
plt.plot(circle_pts[:,0], circle_pts[:,1], 'k--', label='obstacle keypoints')

colors = plt.cm.plasma(np.linspace(0,1,len(last_targets)))

for idx, y_last in enumerate(last_targets):
    # Targets: replace middle keypoint with circle points
    T = np.vstack([
        [x[0], y[0]],
        circle_pts,  # many points
        [3.0, y_last]
    ])
    S_ext = np.vstack([
        [x[0], y[0]],
        np.tile([2.0, 0.6], (len(circle_pts),1)),
        [x[-1], y[-1]]
    ])
    aff = AffineTransform(do_scale=False, do_rotation=True)
    aff.fit(S_ext, T)
    resid = T - aff.predict(S_ext)
    gp = GaussianProcess(kernel=kernel, alpha=1e-10, optimizer=None)
    gp.fit(S_ext, resid)
    def phi_pts(P):
        return aff.predict(P) + gp.predict(P)
    warped = phi_pts(curve)
    plt.plot(warped[:,0], warped[:,1], color=colors[idx], linewidth=1.4,
             label=f"last y={y_last:.2f}" if idx in [0,len(last_targets)-1] else None)

plt.axis('equal'); plt.legend()
plt.title("Obstacle Avoidance")
plt.show()